# Ceview SBERT Training

This notebook trains the tourism multi-class classification model using the consolidated dataset and the 80/10/10 stratified split.

In [ ]:
# Install dependencies
%pip install -r ../requirements.txt

In [ ]:
import sys
import os
import torch

# Add project root to path
sys.path.append(os.path.abspath('..'))

from model.preprocessing import get_stratified_dataloaders
from model.train import train_model
from model.evaluate import evaluate_model
from model.save import save_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 1. Load Data
Load the consolidated JSON dataset and apply the 80/10/10 stratified split.

In [ ]:
dataset_path = '../dataset/consolidated_tourism_data.json'
train_loader, val_loader, test_loader = get_stratified_dataloaders(dataset_path, batch_size=16)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

## 2. Train Model
Run the training loop for 20 epochs. The best model weights will be saved to `saved_models/best_model.pth`.

In [ ]:
model = train_model(train_loader, val_loader, num_epochs=20, learning_rate=0.001, device=device)
save_model(model)

# Automatically download the model if running in Colab
try:
    from google.colab import files
    import os
    # The model is saved to the root of the project
    root = os.path.abspath(os.path.join(os.getcwd(), ".."))
    model_path = os.path.join(root, "saved_models", "best_model.pth")
    if os.path.exists(model_path):
        files.download(model_path)
    else:
        print(f"Could not find model at {model_path}")
except ImportError:
    pass


## 3. Final Unbiased Evaluation
Evaluate the best model against the completely unseen test set.

In [ ]:
import wandb
from model.model import get_model
import torch.nn as nn

# Load the best weights saved during training
best_model = get_model().to(device)
import os
root = os.path.abspath(os.path.join(os.getcwd(), ".."))
model_path = os.path.join(root, "saved_models", "best_model.pth")
best_model.load_state_dict(torch.load(model_path))
best_model.eval()

criterion = nn.BCELoss()
test_loss, test_acc, test_f1 = evaluate_model(best_model, test_loader, criterion, device=device)

print("=== Final Test Set Metrics ===")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test F1 Score (Macro): {test_f1:.4f}")
# Log final test metrics to W&B and close the run
if wandb.run is not None:
    wandb.summary["test_loss"] = test_loss
    wandb.summary["test_accuracy"] = test_acc
    wandb.summary["test_f1_macro"] = test_f1
    wandb.finish()
